In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import sys
sys.path.append("..")

from utils import *
import json 
import pandas as pd
import random
from pathlib import Path
import pickle
from transformers import AutoTokenizer

/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
MODELTYPE = "deepset/gbert-base"
DATASET = "med_indication_all_RF_diag"
DATASETPATH =  Path("../data") / f"ind.{DATASET}"
DATASETFILE = Path("../data") / "medindcls_bert.json"

adj, features, y_train, y_val, y_test, train_mask, val_mask, test_mask, _, _ = load_corpus(DATASETPATH)
doc_mask = train_mask + val_mask + test_mask
adj.sum()

tokenizer = AutoTokenizer.from_pretrained(MODELTYPE)

if not DATASETFILE.exists():
    print("Creating dataset")
    dataset = CleanClinicDataset(tokenizer=tokenizer, clean=False)
    with open(DATASETFILE, "wb") as f:
        print(f"Saving dataset under {DATASETFILE}")
        pickle.dump(dataset, f)
else:
    print(f"Loading dataset from: {DATASETFILE}")
    with open(DATASETFILE, "rb") as f:
        dataset = pickle.load(f)

27996
Loading dataset from: ../data/medindcls_bert.json


/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator LabelEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/pwiesenbach/anaconda3/envs/pytorch/lib/python3.10/site-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator OneHotEncoder from version 0.22 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/tmp/ipykernel_682855/3214339880.py:21: DeprecationWarning: Please use `csr_matrix` from the `scipy.sparse` namespace, the `scipy.sparse.csr` namespace is deprecated.
  dataset = pickle.loa

In [5]:
mixfactor = 0.5
ig_gcn_only_path = f"../models/gcn/{mixfactor}/_ig_attrs_gcn_only.json"
shap_gcn_only_path = f"../models/gcn/{mixfactor}/_shap_values_gcn_only.json"
ig_gcn_bert_path = f"../models/gcn/{mixfactor}/_ig_attrs_gcn_bert.json"
shap_gcn_bert_path = f"../models/gcn/{mixfactor}/_shap_values_gcn_bert.json"

ig_gcn_only_values = pickle.load(open(ig_gcn_only_path, "rb"))
ig_gcn_bert_values = pickle.load(open(ig_gcn_bert_path, "rb"))
shap_gcn_only_values = pickle.load(open(shap_gcn_only_path, "rb"))
shap_gcn_bert_values = pickle.load(open(shap_gcn_bert_path, "rb"))

In [29]:
ig_gcn_only_values[-1]

array([-0.00109209,  0.00087921,  0.00277563, ..., -0.00191309,
        0.0011486 ,  0.34732565])

In [91]:
top_n_interpret = 10
top_ig_gcn_only_values = np.argpartition(ig_gcn_only_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_bert_values = np.argpartition(ig_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_only_values = np.argpartition(shap_gcn_only_values, -top_n_interpret)[:, -top_n_interpret:]
top_shap_gcn_bert_values = np.argpartition(shap_gcn_bert_values, -top_n_interpret)[:, -top_n_interpret:]
top_ig_gcn_only_values[-1]

array([ 649, 2007, 2200, 2617, 1029, 1183, 1208, 1218, 1778, 2698])

In [92]:
random.seed(0)
idx = np.arange(len(dataset))
random.shuffle(idx)

train_len = 1889
val_len = 270
test_len = 540
nb_word = 25297

train_idx, val_idx, test_idx = (
    idx[: int(len(idx) * 0.7)],
    idx[int(len(idx) * 0.7) : int(len(idx) * 0.8)],
    idx[int(len(idx) * 0.8) :],
)

def map_to_idx(x):
    if x < train_len:
        return train_idx[x]
    elif x < val_len + train_len:
        return val_idx[x - train_len]
    else:
        return test_idx[x - val_len - train_len]

In [93]:
top_ig_gcn_only_values = np.vectorize(map_to_idx)(top_ig_gcn_only_values)
top_ig_gcn_bert_values = np.vectorize(map_to_idx)(top_ig_gcn_bert_values)
top_shap_gcn_only_values = np.vectorize(map_to_idx)(top_shap_gcn_only_values)
top_shap_gcn_bert_values = np.vectorize(map_to_idx)(top_shap_gcn_bert_values)

In [94]:
ig_gcn_only_df = pd.DataFrame(top_ig_gcn_only_values, index=test_idx)
ig_gcn_bert_df = pd.DataFrame(top_ig_gcn_bert_values, index=test_idx)
shap_gcn_only_df = pd.DataFrame(top_shap_gcn_only_values, index=test_idx)
shap_gcn_bert_df = pd.DataFrame(top_shap_gcn_bert_values, index=test_idx)

In [95]:
ig_gcn_only_df

,0,1,2,3,4,5,6,7,8,9
2298,2284,2286,2282,2297,2298,2293,2294,2292,2291,2300
2130,820,818,819,821,2132,2135,2136,2130,2133,2131
2559,580,582,589,587,2558,2562,2556,2557,2559,2563
1046,2163,2164,123,1380,1378,2165,1379,122,125,124
1991,1254,1719,1357,1354,1990,1351,1991,1355,1718,1356
...,...,...,...,...,...,...,...,...,...,...
2094,1441,2511,294,292,291,2094,293,1337,1335,1334
1060,1466,1056,1474,1061,1058,1055,1057,1060,1059,1054
165,2149,620,2551,675,1912,1140,676,165,166,619
1722,1731,1734,1721,1728,1730,1735,1736,1726,1722,1732


In [96]:
ig_gcn_only_df = pd.melt(ig_gcn_only_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
ig_gcn_bert_df = pd.melt(ig_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
shap_gcn_only_df = pd.melt(shap_gcn_only_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")
shap_gcn_bert_df = pd.melt(shap_gcn_bert_df,  value_name='rel_id', ignore_index=False).drop(["variable"], axis=1).reset_index(names="id")

In [117]:
labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_only_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_only_df.rel_id])
ig_gcn_only_df["label"] = labels
ig_gcn_only_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in ig_gcn_bert_df.rel_id])
ig_gcn_bert_df["label"] = labels
ig_gcn_bert_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_only_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_only_df.rel_id])
shap_gcn_only_df["label"] = labels
shap_gcn_only_df["rel_label"] = rel_labels

labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.id])
rel_labels = dataset.LE.inverse_transform([dataset[x]["labels"] for x in shap_gcn_bert_df.rel_id])
shap_gcn_bert_df["label"] = labels
shap_gcn_bert_df["rel_label"] = rel_labels

In [118]:
ig_gcn_only_df

,id,rel_id,labels,rel_labels,label,rel_label
0,2298,2284,Blutdrucksenker_beides,Blutdrucksenker_beides,Blutdrucksenker_beides,Blutdrucksenker_beides
1,2130,820,Blutdrucksenker_Herzschw,Blutdrucksenker_Herzschw,Blutdrucksenker_Herzschw,Blutdrucksenker_Herzschw
2,2559,580,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck
3,1046,2163,Cholesterinsenker_unklar,DM_Insulin und Tabletten,Cholesterinsenker_unklar,DM_Insulin und Tabletten
4,1991,1254,Blutdrucksenker_unklar,Blutdrucksenker_unklar,Blutdrucksenker_unklar,Blutdrucksenker_unklar
...,...,...,...,...,...,...
5395,2094,1334,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck
5396,1060,1054,Blutdrucksenker_beides,Blutdrucksenker_beides,Blutdrucksenker_beides,Blutdrucksenker_beides
5397,165,619,Cholesterinsenker_beides,Cholesterinsenker_beides,Cholesterinsenker_beides,Cholesterinsenker_beides
5398,1722,1732,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck,Blutdrucksenker_Blutdruck


In [119]:
ig_gcn_only_source_df = ig_gcn_only_df[["id", "label"]].drop_duplicates()
ig_gcn_only_target_df = ig_gcn_only_df[["rel_id", "rel_label"]].drop_duplicates()
ig_gcn_only_node_df = pd.concat([ig_gcn_only_source_df, ig_gcn_only_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

ig_gcn_bert_source_df = ig_gcn_bert_df[["id", "label"]].drop_duplicates()
ig_gcn_bert_target_df = ig_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
ig_gcn_bert_node_df = pd.concat([ig_gcn_bert_source_df, ig_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

shap_gcn_only_source_df = shap_gcn_only_df[["id", "label"]].drop_duplicates()
shap_gcn_only_target_df = shap_gcn_only_df[["rel_id", "rel_label"]].drop_duplicates()
shap_gcn_only_node_df = pd.concat([shap_gcn_only_source_df, shap_gcn_only_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

shap_gcn_bert_source_df = shap_gcn_bert_df[["id", "label"]].drop_duplicates()
shap_gcn_bert_target_df = shap_gcn_bert_df[["rel_id", "rel_label"]].drop_duplicates()
shap_gcn_bert_node_df = pd.concat([shap_gcn_bert_source_df, shap_gcn_bert_target_df.rename(columns={'rel_id':'id', "rel_label": "label"})], axis=0, ignore_index=True).drop_duplicates()

In [120]:
ig_gcn_only_G=nx.from_pandas_edgelist(ig_gcn_only_df, "id", 'rel_id', create_using=nx.DiGraph)
shap_gcn_only_G=nx.from_pandas_edgelist(shap_gcn_only_df, "id", 'rel_id', create_using=nx.DiGraph)

ig_gcn_bert_G=nx.from_pandas_edgelist(ig_gcn_bert_df, "id", 'rel_id', create_using=nx.DiGraph)
shap_gcn_bert_G=nx.from_pandas_edgelist(shap_gcn_bert_df, "id", 'rel_id', create_using=nx.DiGraph)

In [15]:
ig_gcn_only_id_df = ig_gcn_only_df.iloc[:,0:2]
ig_gcn_only_rel_df = ig_gcn_only_df.iloc[:,2:4]
ig_gcn_bert_id_df = ig_gcn_bert_df.iloc[:,0:2]
ig_gcn_bert_rel_df = ig_gcn_bert_df.iloc[:,2:4]
shap_gcn_only_id_df = shap_gcn_only_df.iloc[:,0:2]
shap_gcn_only_rel_df = shap_gcn_only_df.iloc[:,2:4]
shap_gcn_bert_id_df = shap_gcn_bert_df.iloc[:,0:2]
shap_gcn_bert_rel_df = shap_gcn_bert_df.iloc[:,2:4]

In [16]:
new_columns = ["id", "label"]
ig_gcn_only_id_df.columns = new_columns
ig_gcn_only_rel_df.columns = new_columns
ig_gcn_bert_id_df.columns = new_columns
ig_gcn_bert_rel_df.columns = new_columns
shap_gcn_only_id_df.columns = new_columns
shap_gcn_only_rel_df.columns = new_columns
shap_gcn_bert_id_df.columns = new_columns
shap_gcn_bert_rel_df.columns = new_columns

In [17]:
ig_gcn_only_id_rel_df = pd.concat([ig_gcn_only_id_df, ig_gcn_only_rel_df], ignore_index=True).drop_duplicates()
ig_gcn_only_id2label = dict(zip(ig_gcn_only_id_rel_df.id, ig_gcn_only_id_rel_df.label))
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_id2label, "label")

ig_gcn_bert_id_rel_df = pd.concat([ig_gcn_bert_id_df, ig_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
ig_gcn_bert_id2label = dict(zip(ig_gcn_bert_id_rel_df.id, ig_gcn_bert_id_rel_df.label))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2label, "label")

shap_gcn_only_id_rel_df = pd.concat([shap_gcn_only_id_df, shap_gcn_only_rel_df], ignore_index=True).drop_duplicates()
shap_gcn_only_id2label = dict(zip(shap_gcn_only_id_rel_df.id, shap_gcn_only_id_rel_df.label))
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_id2label, "label")

shap_gcn_bert_id_rel_df = pd.concat([shap_gcn_bert_id_df, shap_gcn_bert_rel_df], ignore_index=True).drop_duplicates()
shap_gcn_bert_id2label = dict(zip(shap_gcn_bert_id_rel_df.id, shap_gcn_bert_id_rel_df.label))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2label, "label")

In [18]:
ig_gcn_only_id2text = {id: dataset.texts[id] for id in ig_gcn_only_id_rel_df.id}
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_id2text, "text")

ig_gcn_bert_id2text = {id: dataset.texts[id] for id in ig_gcn_bert_id_rel_df.id}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2text, "text")

shap_gcn_only_id2text = {id: dataset.texts[id] for id in shap_gcn_only_id_rel_df.id}
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_id2text, "text")

shap_gcn_bert_id2text = {id: dataset.texts[id] for id in shap_gcn_bert_id_rel_df.id}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2text, "text")

In [22]:
ig_gcn_only_id2drug = {node: ig_gcn_only_G.nodes()[node]["text"].split(" ")[1] for node in ig_gcn_only_G.nodes()}
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_id2drug, "drug")

ig_gcn_bert_id2drug = {node: ig_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in ig_gcn_bert_G.nodes()}
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_id2drug, "drug")

shap_gcn_only_id2drug = {node: shap_gcn_only_G.nodes()[node]["text"].split(" ")[1] for node in shap_gcn_only_G.nodes()}
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_id2drug, "drug")

shap_gcn_bert_id2drug = {node: shap_gcn_bert_G.nodes()[node]["text"].split(" ")[1] for node in shap_gcn_bert_G.nodes()}
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_id2drug, "drug")

In [23]:
#nx.is_connected(ig_gcn_only_G)
#nx.number_connected_components(G)

In [24]:
#components = nx.connected_components(G)
#largest_component = max(components, key=len)
#subgraph = G.subgraph(largest_component)
#diameter = nx.diameter(subgraph)
#print("Network diameter of largest component:", diameter)

In [25]:
ig_gcn_only_triadic_closure = nx.transitivity(ig_gcn_only_G)
ig_gcn_bert_triadic_closure = nx.transitivity(ig_gcn_bert_G)
shap_gcn_only_triadic_closure = nx.transitivity(shap_gcn_only_G)
shap_gcn_bert_triadic_closure = nx.transitivity(shap_gcn_bert_G)
ig_gcn_only_triadic_closure, ig_gcn_bert_triadic_closure, shap_gcn_only_triadic_closure, shap_gcn_bert_triadic_closure

(0.11742732262655012,
 0.10746515592098667,
 0.08615833417161252,
 0.08531496656779448)

In [26]:
ig_gcn_only_degree_dict = dict(ig_gcn_only_G.degree(ig_gcn_only_G.nodes()))
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_degree_dict, 'degree')
ig_gcn_only_sorted_degree = sorted(ig_gcn_only_degree_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_degree_dict = dict(ig_gcn_bert_G.degree(ig_gcn_bert_G.nodes()))
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_degree_dict, 'degree')
ig_gcn_bert_sorted_degree = sorted(ig_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_only_degree_dict = dict(shap_gcn_only_G.degree(shap_gcn_only_G.nodes()))
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_degree_dict, 'degree')
shap_gcn_only_sorted_degree = sorted(shap_gcn_only_degree_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_degree_dict = dict(shap_gcn_bert_G.degree(shap_gcn_bert_G.nodes()))
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_degree_dict, 'degree')
shap_gcn_bert_sorted_degree = sorted(shap_gcn_bert_degree_dict.items(), key=itemgetter(1), reverse=True)

#for node, degree in shap_sorted_degree[:10]:
#    print(shap_G.nodes()[node]["drug"])

ig_gcn_only_sorted_degree[:10], ig_gcn_bert_sorted_degree[:10], shap_gcn_only_sorted_degree[:10], shap_gcn_bert_sorted_degree[:10]

([(1912, 27),
  (2160, 25),
  (2164, 22),
  (1824, 22),
  (2466, 22),
  (1372, 21),
  (0, 20),
  (1270, 20),
  (1745, 20),
  (678, 20)],
 [(1912, 24),
  (1354, 24),
  (628, 22),
  (2466, 22),
  (191, 21),
  (2029, 21),
  (0, 20),
  (1577, 20),
  (2598, 20),
  (1835, 20)],
 [(1912, 23),
  (1577, 23),
  (2466, 23),
  (1275, 22),
  (191, 21),
  (187, 21),
  (2025, 21),
  (1786, 21),
  (254, 20),
  (1953, 20)],
 [(1912, 27),
  (1786, 23),
  (254, 21),
  (2466, 21),
  (628, 20),
  (1577, 20),
  (66, 20),
  (1066, 19),
  (1783, 19),
  (375, 19)])

In [27]:
ig_gcn_only_betweenness_dict = nx.betweenness_centrality(ig_gcn_only_G)
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_betweenness_dict, 'betweenness')
ig_gcn_only_sorted_betweenness = sorted(ig_gcn_only_betweenness_dict.items(), key=itemgetter(1), reverse=True)

ig_gcn_bert_betweenness_dict = nx.betweenness_centrality(ig_gcn_bert_G)
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_betweenness_dict, 'betweenness')
ig_gcn_bert_sorted_betweenness = sorted(ig_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_only_betweenness_dict = nx.betweenness_centrality(shap_gcn_only_G)
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_betweenness_dict, 'betweenness')
shap_gcn_only_sorted_betweenness = sorted(shap_gcn_only_betweenness_dict.items(), key=itemgetter(1), reverse=True)

shap_gcn_bert_betweenness_dict = nx.betweenness_centrality(shap_gcn_bert_G)
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_betweenness_dict, 'betweenness')
shap_gcn_bert_sorted_betweenness = sorted(shap_gcn_bert_betweenness_dict.items(), key=itemgetter(1), reverse=True)


#for node, degree in shap_sorted_betweenness[:10]:
#    print(shap_G.nodes()[node]["drug"])
#for node, degree in ig_sorted_betweenness[:10]:
#    print(ig_G.nodes()[node]["drug"])
ig_gcn_only_sorted_betweenness[:10], ig_gcn_bert_sorted_betweenness[:10], shap_gcn_only_sorted_betweenness[:10], shap_gcn_bert_sorted_betweenness[:10]

([(382, 0.018606077796619703),
  (2160, 0.01666786948577872),
  (1970, 0.014512346805948043),
  (1352, 0.014452026152336808),
  (383, 0.014196680934495642),
  (1875, 0.0135023340668173),
  (0, 0.013205221860144552),
  (66, 0.013160836959417426),
  (1913, 0.013031534927175807),
  (1242, 0.012040922343300807)],
 [(1875, 0.03349577234886378),
  (0, 0.03132064174338501),
  (1550, 0.028728158728052823),
  (66, 0.027099548719250143),
  (1913, 0.021719654201521728),
  (1352, 0.020896158828074853),
  (1548, 0.0175758667793998),
  (1746, 0.016796458600848628),
  (2160, 0.016049357500855108),
  (2029, 0.015409568110510013)],
 [(2263, 0.02049515893507008),
  (1550, 0.02028695497689331),
  (382, 0.020043652836311324),
  (255, 0.01962404715890004),
  (1026, 0.016367328132609667),
  (1746, 0.015154907430389334),
  (2025, 0.013970524211260497),
  (383, 0.013610175523892492),
  (2456, 0.013580673698649555),
  (772, 0.013296230664658428)],
 [(1912, 0.023388482505417144),
  (1468, 0.02147475566193704),


In [28]:
ig_gcn_only_communities = community.greedy_modularity_communities(ig_gcn_only_G)
ig_gcn_only_modularity_dict = {}
for i, c in enumerate(ig_gcn_only_communities):
    for name in c:
        ig_gcn_only_modularity_dict[name] = i
nx.set_node_attributes(ig_gcn_only_G, ig_gcn_only_modularity_dict, 'community')

ig_gcn_bert_communities = community.greedy_modularity_communities(ig_gcn_bert_G)
ig_gcn_bert_modularity_dict = {}
for i, c in enumerate(ig_gcn_bert_communities):
    for name in c:
        ig_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(ig_gcn_bert_G, ig_gcn_bert_modularity_dict, 'community')

shap_gcn_only_communities = community.greedy_modularity_communities(shap_gcn_only_G)
shap_gcn_only_modularity_dict = {}
for i, c in enumerate(shap_gcn_only_communities):
    for name in c:
        shap_gcn_only_modularity_dict[name] = i
nx.set_node_attributes(shap_gcn_only_G, shap_gcn_only_modularity_dict, 'community')

shap_gcn_bert_communities = community.greedy_modularity_communities(shap_gcn_bert_G)
shap_gcn_bert_modularity_dict = {}
for i, c in enumerate(shap_gcn_bert_communities):
    for name in c:
        shap_gcn_bert_modularity_dict[name] = i
nx.set_node_attributes(shap_gcn_bert_G, shap_gcn_bert_modularity_dict, 'community')

len(ig_gcn_only_communities), len(ig_gcn_bert_communities), len(shap_gcn_only_communities), len(shap_gcn_bert_communities)

(36, 34, 31, 31)

In [29]:
([Counter([ig_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_only_communities[:10]],
 [Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_only_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0][1]/len(com) for com in shap_gcn_bert_communities[:10]])

([0.2,
  0.5645161290322581,
  0.41904761904761906,
  0.4368932038834951,
  0.25,
  0.5666666666666667,
  0.5783132530120482,
  0.3780487804878049,
  0.5569620253164557,
  0.358974358974359],
 [0.26666666666666666,
  0.6952380952380952,
  0.48,
  0.46464646464646464,
  0.6458333333333334,
  0.43333333333333335,
  0.367816091954023,
  0.7023809523809523,
  0.2625,
  0.631578947368421],
 [0.6170212765957447,
  0.3140495867768595,
  0.358974358974359,
  0.46601941747572817,
  0.297029702970297,
  0.39603960396039606,
  0.6831683168316832,
  0.36363636363636365,
  0.8,
  0.328125],
 [0.2727272727272727,
  0.6046511627906976,
  0.33064516129032256,
  0.5543478260869565,
  0.5222222222222223,
  0.5287356321839081,
  0.49411764705882355,
  0.425,
  0.5189873417721519,
  0.717948717948718])

In [30]:
([Counter([ig_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in ig_gcn_only_communities[:10]],
 [Counter([ig_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in ig_gcn_bert_communities[:10]],
 [Counter([shap_gcn_only_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in shap_gcn_only_communities[:10]],
 [Counter([shap_gcn_bert_G.nodes()[id]["label"] for id in com]).most_common()[0] for com in shap_gcn_bert_communities[:10]])

([('Blutdrucksenker_beides', 25),
  ('Blutdrucksenker_Blutdruck', 70),
  ('Blutdrucksenker_beides', 44),
  ('DM_nur Tabletten', 45),
  ('Cholesterinsenker_KHK', 24),
  ('Blutdrucksenker_beides', 51),
  ('Blutdrucksenker_beides', 48),
  ('Blutdrucksenker_Blutdruck', 31),
  ('Blutdrucksenker_Blutdruck', 44),
  ('Blutdrucksenker_beides', 28)],
 [('Blutdrucksenker_unklar', 32),
  ('Blutdrucksenker_Blutdruck', 73),
  ('Blutdrucksenker_Blutdruck', 48),
  ('Blutdrucksenker_beides', 46),
  ('Blutdrucksenker_beides', 62),
  ('Cholesterinsenker_beides', 39),
  ('Blutdrucksenker_beides', 32),
  ('Blutdrucksenker_Blutdruck', 59),
  ('Blutdrucksenker_beides', 21),
  ('Blutdrucksenker_beides', 48)],
 [('Blutdrucksenker_beides', 87),
  ('DM_nur Tabletten', 38),
  ('Cholesterinsenker_beides', 42),
  ('Blutdrucksenker_beides', 48),
  ('Blutdrucksenker_Herzschw', 30),
  ('Cholesterinsenker_beides', 40),
  ('Blutdrucksenker_Blutdruck', 69),
  ('Blutdrucksenker_unklar', 36),
  ('Blutdrucksenker_Blutdruck'